# RAG Chain & Prompt Engineering

## Overview

This document explains how the prompt engineering layer works — the part that turns retrieved document chunks into a well-structured prompt for the LLM. It also covers the full RAG pipeline: embed the question → search Qdrant → build prompt → send to LLM → get answer.

This is where the "intelligence" of the system lives. The LLM doesn't know anything about your documents — it only knows what you put in the prompt. Good prompt engineering means grounded, cited answers. Bad prompt engineering means hallucination.

## Architecture Context

In the chat service, this logic lives in `prompt.py` (templates) and `chain.py` (the pipeline). The `/chat` endpoint calls `rag_query()` which orchestrates: embed → search → prompt → stream.

In [ ]:
# Prerequisites — requires Ollama (mistral + nomic-embed-text) and Qdrant
import httpx
import numpy as np

OLLAMA_BASE_URL = "http://localhost:11434"
QDRANT_HOST = "localhost"
QDRANT_PORT = 6333

async def check_services():
    # Check Ollama
    try:
        async with httpx.AsyncClient() as client:
            resp = await client.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5.0)
            models = [m["name"] for m in resp.json().get("models", [])]
            has_mistral = any("mistral" in m for m in models)
            has_embed = any("nomic-embed-text" in m for m in models)
            print(f"✓ Ollama: mistral={'✓' if has_mistral else '✗'}, nomic-embed-text={'✓' if has_embed else '✗'}")
    except Exception as e:
        print(f"✗ Ollama: {e}")

    # Check Qdrant
    try:
        from qdrant_client import QdrantClient
        client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)
        client.get_collections()
        print(f"✓ Qdrant connected")
    except Exception as e:
        print(f"✗ Qdrant: {e}")

await check_services()

## Package Introductions

No new packages in this document — it combines httpx (from `03_ollama_embeddings`) and qdrant-client (from `04_qdrant_vector_storage`). The focus here is on **prompt design**, which is about crafting text, not calling APIs.

The one new concept is **prompt templates** — string formatting patterns that structure how context and questions are presented to the LLM. These are just Python f-strings or `.format()` calls. No library needed.

## Go/TS Comparison

Prompt engineering is language-agnostic — it's about the text you send to the LLM, not the code. The closest analogy in traditional web development is **template rendering** (like Go's `html/template` or Handlebars in JS). You have a template with placeholders, you fill in data, and the result goes to the consumer (LLM instead of browser).

The RAG pattern itself is similar to how you'd build a search feature:
1. User submits query → **embed** (like building a search index query)
2. Search the database → **retrieve** (like `SELECT ... WHERE similarity > threshold`)
3. Format results for display → **prompt** (like rendering a template)
4. Return to user → **generate** (except the LLM generates natural language, not HTML)

## Build It

### Step 1: Define the prompt templates

These templates control what the LLM sees. The system prompt sets the LLM's behavior. The RAG template structures how context and questions are presented.

In [ ]:
SYSTEM_PROMPT = """You are a helpful document Q&A assistant. Answer questions based only on the provided context. If the context doesn't contain enough information to answer, say so honestly — do not make up information.

When referencing information, mention the source file and page number."""

RAG_TEMPLATE = """Context:
{context}

Question: {question}

Answer based only on the context above. Cite sources (filename, page) when possible."""

NO_CONTEXT_TEMPLATE = """The user asked: {question}

I don't have any relevant context from uploaded documents to answer this question. Please upload a relevant document first, or rephrase your question."""


def build_rag_prompt(question: str, chunks: list[dict]) -> str:
    """Build a prompt from retrieved chunks and a question."""
    if not chunks:
        return NO_CONTEXT_TEMPLATE.format(question=question)

    context_parts = []
    for chunk in chunks:
        source = f"[{chunk['filename']}, page {chunk['page_number']}]"
        context_parts.append(f"{source}\n{chunk['text']}")

    context = "\n\n".join(context_parts)
    return RAG_TEMPLATE.format(context=context, question=question)

# Test with sample chunks
sample_chunks = [
    {"text": "Revenue was 2.5 million dollars.", "filename": "report.pdf", "page_number": 1},
    {"text": "Operating margins improved to 23%.", "filename": "report.pdf", "page_number": 1},
]

prompt = build_rag_prompt("What was the revenue?", sample_chunks)
print("=== Generated Prompt ===")
print(prompt)

In [ ]:
# Test with no chunks
prompt = build_rag_prompt("What is quantum computing?", [])
print("=== No Context Prompt ===")
print(prompt)

### Step 2: Wire up the full RAG pipeline

Now the full pipeline comes together:
1. Seed Qdrant with some document chunks (reusing code from `02_pdf_parsing_and_chunking` through `04_qdrant_vector_storage`)
2. Embed a question
3. Search Qdrant for similar chunks
4. Build a prompt with the results
5. Send it to Ollama and get an answer

In [ ]:
# First, let's seed Qdrant with real embeddings
# (Combining what we built in Lessons 02, 03, and 04)

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams
import uuid

COLLECTION = "lesson_05_test"

# Embed function from Lesson 03
async def embed_texts(texts, ollama_base_url, model):
    if not texts:
        return []
    async with httpx.AsyncClient() as client:
        response = await client.post(
            f"{ollama_base_url}/api/embed",
            json={"model": model, "input": texts},
            timeout=120.0,
        )
        response.raise_for_status()
        return response.json()["embeddings"]

# Create collection and seed with chunks
qdrant = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)
if qdrant.collection_exists(COLLECTION):
    qdrant.delete_collection(COLLECTION)

qdrant.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=768, distance=Distance.COSINE),
)

chunks = [
    {"text": "The quarterly revenue was 2.5 million dollars, up 15% from last year.", "page_number": 1, "chunk_index": 0},
    {"text": "Operating margins improved to 23%, driven by cost optimization.", "page_number": 1, "chunk_index": 1},
    {"text": "The engineering team grew to 45 people. Three new products launched.", "page_number": 2, "chunk_index": 2},
    {"text": "Customer satisfaction scores reached 4.8 out of 5.", "page_number": 2, "chunk_index": 3},
    {"text": "Infrastructure costs decreased by 12% due to cloud migration.", "page_number": 2, "chunk_index": 4},
]

texts = [c["text"] for c in chunks]
vectors = await embed_texts(texts, OLLAMA_BASE_URL, "nomic-embed-text")

points = [
    PointStruct(
        id=str(uuid.uuid4()),
        vector=vec,
        payload={"text": c["text"], "page_number": c["page_number"],
                 "filename": "q3_report.pdf", "document_id": "doc-001",
                 "chunk_index": c["chunk_index"]},
    )
    for c, vec in zip(chunks, vectors)
]
qdrant.upsert(collection_name=COLLECTION, points=points)
print(f"Seeded {len(points)} chunks into '{COLLECTION}'")

In [ ]:
# Now: the full RAG pipeline

async def rag_answer(question: str) -> str:
    """Full RAG pipeline: embed → search → prompt → generate."""

    # 1. Embed the question
    q_vectors = await embed_texts([question], OLLAMA_BASE_URL, "nomic-embed-text")
    q_vector = q_vectors[0]

    # 2. Search Qdrant
    results = qdrant.search(
        collection_name=COLLECTION,
        query_vector=q_vector,
        limit=3,
    )
    retrieved_chunks = [
        {"text": hit.payload["text"],
         "filename": hit.payload["filename"],
         "page_number": hit.payload["page_number"],
         "score": hit.score}
        for hit in results
    ]

    print("Retrieved chunks:")
    for c in retrieved_chunks:
        print(f"  [{c['score']:.3f}] {c['text'][:60]}...")

    # 3. Build prompt
    prompt = build_rag_prompt(question, retrieved_chunks)

    # 4. Send to Ollama (non-streaming for simplicity)
    async with httpx.AsyncClient() as client:
        response = await client.post(
            f"{OLLAMA_BASE_URL}/api/generate",
            json={"model": "mistral", "prompt": prompt, "system": SYSTEM_PROMPT, "stream": False},
            timeout=120.0,
        )
        response.raise_for_status()
        return response.json()["response"]

# Try it!
answer = await rag_answer("What was the company's revenue?")
print(f"\n=== Answer ===\n{answer}")

## Experiment

In [ ]:
# Experiment 1: Hallucination demo — ask about something NOT in the context
answer = await rag_answer("What is the CEO's name?")
print(f"Answer: {answer}")
print("\nThe LLM should say it doesn't have enough context.")
print("If it makes up a name, the prompt engineering needs work.")

In [ ]:
# Experiment 2: What happens WITHOUT grounding instructions?
BAD_TEMPLATE = """Here is some context: {context}

Question: {question}

Answer the question."""

def build_bad_prompt(question, chunks):
    context = "\n".join(c["text"] for c in chunks)
    return BAD_TEMPLATE.format(context=context, question=question)

# Ask the same question with a weaker prompt
q_vecs = await embed_texts(["What is the CEO's salary?"], OLLAMA_BASE_URL, "nomic-embed-text")
results = qdrant.search(collection_name=COLLECTION, query_vector=q_vecs[0], limit=3)
chunks_for_prompt = [{"text": h.payload["text"], "filename": h.payload["filename"], "page_number": h.payload["page_number"]} for h in results]
bad_prompt = build_bad_prompt("What is the CEO's salary?", chunks_for_prompt)

async with httpx.AsyncClient() as client:
    resp = await client.post(
        f"{OLLAMA_BASE_URL}/api/generate",
        json={"model": "mistral", "prompt": bad_prompt, "stream": False},
        timeout=120.0,
    )
    print(f"Without grounding: {resp.json()['response']}")
    print("\nNotice: without explicit instructions to only use context,")
    print("the LLM is more likely to hallucinate or speculate.")

In [ ]:
# Experiment 3: Change the system prompt personality
CASUAL_SYSTEM = "You are a casual, friendly assistant. Use simple language and emoji when appropriate. Answer based only on the provided context."

async with httpx.AsyncClient() as client:
    prompt = build_rag_prompt("How much revenue did they make?", chunks_for_prompt)
    resp = await client.post(
        f"{OLLAMA_BASE_URL}/api/generate",
        json={"model": "mistral", "prompt": prompt, "system": CASUAL_SYSTEM, "stream": False},
        timeout=120.0,
    )
    print(f"Casual style: {resp.json()['response']}")

In [ ]:
# Clean up
qdrant.delete_collection(COLLECTION)
print("Cleaned up test collection")

## Check Your Understanding

1. **What is "grounding" in prompt engineering, and why is it critical for RAG applications?**

2. **Why do we include source attribution instructions in the system prompt? What would the user experience be without them?**

3. **The `build_rag_prompt` function has a separate template for when no chunks are found. Why not just send the question to the LLM directly?**